# RESEARCH-GRADE ICA PREPROCESSING PIPELINE (FIXED)
### Tinnitus EEG Phenotyping — Zenodo EEG-PRO (Record 13219018)

--- 
### SCIENTIFIC RATIONALE
This pipeline addresses the primary reviewer critique regarding potential artifactual contributions. It uses conservative ICLabel rejection to ensure that phenotypes are neurological, not artifactual.

### FIX LOG
- **v2.1:** Fixed `e1` scoping bug in `load_eeglab_file`. Added `err_s1/s2/s3` persistent variables to capture why loading fails.

In [ ]:
# %% ── CELL 1: INSTALL DEPENDENCIES ──────────────────────────────────────────
!pip install -q mne==1.7.1 mne-icalabel pyriemann zenodo_get pymatreader tqdm
import mne
print(f"MNE version: {mne.__version__}")

In [ ]:
# %% ── CELL 2: MOUNT GOOGLE DRIVE ────────────────────────────────────────────
from google.colab import drive
import os
drive.mount('/content/drive')

BASE      = '/content/drive/MyDrive/Tinnitus_ICA'
RAW_DIR   = '/content/eeg_data/EEG-PRO_data/RAW/Tinnitus Dataset'
CLEAN_DIR = os.path.join(BASE, 'ica_cleaned')
LOG_DIR   = os.path.join(BASE, 'logs')

for d in [CLEAN_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Results → {BASE}")

In [ ]:
# %% ── CELL 3: INVENTORY RAW DATA ────────────────────────────────────────────
import glob
set_files = sorted(glob.glob(os.path.join(RAW_DIR, '**', '*.set'), recursive=True))
print(f"Found {len(set_files)} Tinnitus subjects.")

In [ ]:
# %% ── CELL 4: CONFIGURATION ─────────────────────────────────────────────────
CFG = {
    'min_channels'    : 120,    'min_duration_s'  : 30.0,
    'hp_freq'         : 1.0,    'lp_freq'         : 45.0,
    'ica_method'      : 'fastica', 'ica_n_components': 0.99,
    'ica_max_iter'    : 500,    'ica_random_state': 42,
    'reject_labels'   : ['eye', 'muscle'],
    'reject_threshold': 0.80,
    'clean_dir'       : CLEAN_DIR,
    'log_path'        : os.path.join(LOG_DIR, 'ica_audit_log.csv'),
}
print("Config loaded.")

In [ ]:
# %% ── CELL 5: HELPER FUNCTIONS (FIXED) ───────────────────────────────────────
import mne, numpy as np, gc
from mne.preprocessing import ICA
from mne_icalabel import label_components

def load_eeglab_file(file_path):
    err_s1 = err_s2 = err_s3 = "None"
    try:
        return mne.io.read_raw_eeglab(file_path, preload=True, verbose=False), 'Raw'
    except Exception as e: err_s1 = str(e)
    try:
        return mne.io.read_epochs_eeglab(file_path, verbose=False), 'Epochs'
    except Exception as e: err_s2 = str(e)
    try:
        from pymatreader import read_mat
        mat = read_mat(file_path)
        eeg = mat.get('EEG', mat)
        data = np.array(eeg['data'], dtype=np.float64)
        sfreq = float(eeg.get('srate', 128.0))
        ch_names = [str(c.get('labels', f'E{i+1}')) if isinstance(c, dict) else str(c) for i, c in enumerate(eeg.get('chanlocs', []))]
        if len(ch_names) != (data.shape[0] if data.ndim != 3 else data.shape[1]):
            ch_names = [f'E{i+1}' for i in range(data.shape[0] if data.ndim != 3 else data.shape[1])]
        if data.ndim == 3:
            data = np.transpose(data, (2, 0, 1))
            return mne.EpochsArray(data, mne.create_info(ch_names, sfreq, 'eeg'), verbose=False), 'Epochs'
        return mne.io.RawArray(data, mne.create_info(ch_names, sfreq, 'eeg'), verbose=False), 'Raw'
    except Exception as e: err_s3 = str(e)
    raise RuntimeError(f"Load failed for {os.path.basename(file_path)}: S1={err_s1[:50]}, S2={err_s2[:50]}, S3={err_s3[:50]}")

def validate_signal(data, cfg):
    n_ch = len(data.ch_names)
    dur = data.times[-1] if hasattr(data, 'times') else len(data) * data.times[-1]
    if n_ch < cfg['min_channels']: return False, f"FAIL_CH: {n_ch}<{cfg['min_channels']}"
    if dur < cfg['min_duration_s']: return False, f"FAIL_DUR: {dur:.1f}s<{cfg['min_duration_s']}s"
    return True, 'OK'

def run_ica_and_clean(data, data_type, cfg):
    if data_type == 'Epochs':
        data = mne.io.RawArray(mne.concatenate_epochs([data]).get_data().reshape(len(data.ch_names), -1), data.info, verbose=False)
    sfreq = data.info['sfreq']
    filter_len = 'auto'
    if data.n_times <= int(3.3 / cfg['hp_freq'] * sfreq):
        filter_len = f"{data.n_times - 1}" if data.n_times > 1 else 'auto'
    data.filter(cfg['hp_freq'], cfg['lp_freq'], method='fir', filter_length=filter_len, verbose=False)
    data.set_eeg_reference('average', projection=True, verbose=False).apply_proj(verbose=False)
    ica = ICA(n_components=cfg['ica_n_components'], method=cfg['ica_method'], max_iter=cfg['ica_max_iter'], random_state=cfg['ica_random_state'])
    ica.fit(data, verbose=False)
    try: data.set_montage(mne.channels.make_standard_montage('GSN-HydroCel-128'), on_missing='ignore', verbose=False)
    except: pass
    ic_labels = label_components(data, ica, method='iclabel')
    exclude = [i for i, (l, p) in enumerate(zip(ic_labels['labels'], ic_labels['y_pred_proba'])) if l in cfg['reject_labels'] and p.max() > cfg['reject_threshold']]
    ica.exclude = exclude
    return ica.apply(data.copy(), verbose=False), len([i for i, l in enumerate(ic_labels['labels']) if i in exclude and l == 'eye']), len([i for i, l in enumerate(ic_labels['labels']) if i in exclude and l == 'muscle']), ica.n_components_

In [ ]:
# %% ── CELL 6: MAIN LOOP ──────────────────────────────────────────────────────
import csv, glob, os
from tqdm.auto import tqdm
counters = {'ok': 0, 'skipped': 0, 'error': 0}
with open(CFG['log_path'], 'a', newline='') as f:
    writer = csv.writer(f)
    if os.stat(CFG['log_path']).st_size == 0: writer.writerow(['subject', 'ch', 'dur', 'eye', 'musc', 'status', 'notes'])
    for file_path in tqdm(set_files, desc="ICA Pipeline"):
        subj = os.path.basename(file_path)
        out  = os.path.join(CFG['clean_dir'], subj.replace('.set', '-ica-raw.fif'))
        if os.path.exists(out): continue
        try:
            data, dtype = load_eeglab_file(file_path)
            passed, reason = validate_signal(data, CFG)
            if not passed: writer.writerow([subj, len(data.ch_names), 0, 0, 0, reason, '']); del data; counters['skipped']+=1; continue
            clean, ne, nm, nc = run_ica_and_clean(data, dtype, CFG)
            clean.save(out, overwrite=True, verbose=False)
            writer.writerow([subj, len(data.ch_names), clean.times[-1], ne, nm, 'OK', ''])
            counters['ok']+=1; del data, clean; gc.collect(); f.flush()
        except Exception as e:
            writer.writerow([subj, 0, 0, 0, 0, 'ERROR', str(e)[:100]]); counters['error']+=1; f.flush()
print(f"\nCleaned: {counters['ok']} | Errors: {counters['error']}")